In [1]:
!pip install transformers torch seqeval sentencepiece -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [2]:
from google.colab import files
uploaded = files.upload()  # 选择 test_run1.json 上传

Saving test_run1.json to test_run1.json


# Run 1 — v3
## 思路说明



**背景**
本脚本对 Torres Aguilar (2022) 发布的预训练模型（`magistermilitum/roberta-multilingual-medieval-ner`）进行零样本测试（zero-shot evaluation），模型只识别 `PERS`（人名）和 `LOC`（地名）两类实体。我们的 gold 标注体系有5类：`auteur`、`ouvrage`、`material`、`date`、`etat`。唯一有意义的对应关系：`auteur → PERS`。

**三种聚合策略对比**
同一模型跑三次，比较 `simple` / `first` / `max` 三种 subword token 聚合方式的差异。

**改动1 — `first` 策略空格修复**
`aggregation_strategy="first"` 在拼接 subword tokens 时会丢失空格（如 `LeonisUrbevetani`）。
修复方法：用预测结果中的字符偏移量（`start`/`end`）从原文切片，再用 `" ".join(fragment.split())` 重建，得到 `Leonis Urbevetani`。

**改动2 — PERS 预测细分（auteur vs ouvrage 里的人名）**
模型预测出 `PERS` 时无法区分"作者"和"作品名里的人名"。
我们通过 span 位置重叠（overlap）将预测细分为两类：
- `pred_pers_auteur`：与 gold `auteur` span 位置重叠的预测 → 对应主任务
- `pred_pers_ouvrage`：与 gold `ouvrage` 中标注的人名位置重叠的预测 → 量化结构性假阳性

两列互斥：同一预测只属于其中一类。
ouvrage 里的人名来自 `test_run1.json` 中的 `contains_persons` 字段（经 VIAF/Wikidata 人工核查标注）。

**改动3 — 分别计算两套 F1**
- `F1_auteur`：pred PERS vs gold auteur → 主任务指标
- `F1_ouvrage_pers`：pred PERS vs gold ouvrage 里的人名 → 假阳性来源量化

**Bug 修正（v2 → v3）**
v2 中 `pred_auteur` 列实为所有 PERS 预测，未真正过滤。v3 中两列严格互斥：
只有与 gold auteur 位置重叠的预测才进入 `pred_auteur`，
只有与 gold ouvrage 人名位置重叠的预测才进入 `pred_ouvrage`。

---



**Contexte**
Ce script évalue en mode zero-shot le modèle pré-entraîné de Torres Aguilar (2022) (`magistermilitum/roberta-multilingual-medieval-ner`), qui ne reconnaît que `PERS` et `LOC`. Notre gold standard comporte 5 labels : `auteur`, `ouvrage`, `material`, `date`, `etat`. La seule correspondance valide est `auteur → PERS`.

**Comparaison de trois stratégies d'agrégation**
Le même modèle est exécuté trois fois pour comparer `simple`, `first` et `max`.

**Amélioration 1 — correction des espaces pour `first`**
`aggregation_strategy="first"` concatène les subwords sans espace (ex. `LeonisUrbevetani`).
Correction : reconstruction depuis les offsets `start`/`end` dans le texte brut, puis `" ".join(fragment.split())` → `Leonis Urbevetani`.

**Amélioration 2 — distinction auteur / personnes dans l'ouvrage**
Le modèle prédit `PERS` sans distinguer auteur et personne citée dans un titre d'ouvrage.
On divise les prédictions en deux catégories par chevauchement de spans :
- `pred_pers_auteur` : préd. chevauchant un span `auteur` gold → tâche principale
- `pred_pers_ouvrage` : préd. chevauchant les personnes annotées dans `ouvrage` → faux positifs structurels

Les deux colonnes sont mutuellement exclusives.
Les personnes dans les ouvrages proviennent du champ `contains_persons` de `test_run1.json` (vérifiées manuellement via VIAF/Wikidata — voir `justification_ouvrage_persons_run1.md`).

**Amélioration 3 — deux F1 distincts**
- `F1_auteur` : pred PERS vs gold auteur → indicateur principal
- `F1_ouvrage_pers` : pred PERS vs gold personnes dans ouvrage → quantification des faux positifs

**Correction de bug (v2 → v3)**
Dans v2, la colonne `pred_auteur` contenait toutes les prédictions PERS sans filtrage réel.
Dans v3, les deux colonnes sont strictement exclusives : une prédiction n'apparaît que dans la colonne dont le span gold chevauche sa position.


In [3]:
# ════════════════════════════════════════════════════════════════
# Cell 1 — Imports et configuration
# 导入库 + 配置路径和日志
# ════════════════════════════════════════════════════════════════

import json, os, numpy as np
from datetime import datetime

import torch
from transformers import pipeline
from seqeval.metrics import classification_report, f1_score, precision_score, recall_score

MODEL_NAME = "magistermilitum/roberta-multilingual-medieval-ner"
TEST_PATH  = "/content/test_run1.json"   # gold 文件（含 contains_persons 字段）
LOG_DIR    = "/content/run1_v3"
os.makedirs(LOG_DIR, exist_ok=True)

_LOG_PATH  = os.path.join(LOG_DIR, "output.log")
_log_lines = []

def _log(msg=""):
    print(msg)
    _log_lines.append(str(msg))

def _save_log():
    with open(_LOG_PATH, "w", encoding="utf-8") as f:
        f.write("\n".join(_log_lines) + "\n")

device = 0 if torch.cuda.is_available() else -1
_log(f"[Config] Device : {'GPU' if device==0 else 'CPU'}")
_log(f"[Config] Modèle : {MODEL_NAME}")
_log(f"[Config] Données: {TEST_PATH}")


[Config] Device : GPU
[Config] Modèle : magistermilitum/roberta-multilingual-medieval-ner
[Config] Données: /content/test_run1.json


In [4]:
# ════════════════════════════════════════════════════════════════
# Cell 2 — Chargement des données
# 加载 test_run1.json，显示 auteur 和 ouvrage 里的人名
# ════════════════════════════════════════════════════════════════

with open(TEST_PATH, encoding="utf-8") as f:
    test_data = json.load(f)
_log(f"  {len(test_data)} entrées chargées")

for e in test_data:
    auteurs  = [s["text"] for s in e.get("spans", []) if s["label"] == "auteur"]
    ouv_pers = [
        (s["text"], s.get("contains_persons", []))
        for s in e.get("spans", [])
        if s.get("label_detail") == "ouvrage_with_pers"
    ]
    if auteurs or ouv_pers:
        _log(f"  #{e['entry_id']}  auteurs={auteurs}  ouvrage_avec_pers={ouv_pers}")


  10 entrées chargées
  #1298  auteurs=['da Cascia Fra Simone']  ouvrage_avec_pers=[]
  #276  auteurs=['Hieronymi', 'Origenis']  ouvrage_avec_pers=[]
  #333  auteurs=[]  ouvrage_avec_pers=[('Vitae S. Ioannis Gualberti Abbatis, S. Bernardi et aliorum', ['Ioannis Gualberti', 'Bernardi'])]
  #338  auteurs=['Leonis Urbevetani']  ouvrage_avec_pers=[]
  #351  auteurs=['Basilii', 'Marsilii Ficini']  ouvrage_avec_pers=[]
  #235  auteurs=['Origenes']  ouvrage_avec_pers=[]
  #286  auteurs=['S. Ephraem', 'Chrysostomi']  ouvrage_avec_pers=[]
  #1232  auteurs=['Boccaccii']  ouvrage_avec_pers=[('Eclogae ad Donatum Apenninigenam', ['Donatum Apenninigenam'])]


In [5]:
# ════════════════════════════════════════════════════════════════
# Cell 3 — Chargement du modèle (3 stratégies)
# 加载模型，3种聚合策略：simple / first / max
# ════════════════════════════════════════════════════════════════

ner_pipes = {
    "simple": pipeline("token-classification", model=MODEL_NAME,
                       aggregation_strategy="simple", device=device),
    "first":  pipeline("token-classification", model=MODEL_NAME,
                       aggregation_strategy="first",  device=device),
    "max":    pipeline("token-classification", model=MODEL_NAME,
                       aggregation_strategy="max",    device=device),
}
print("  Modèles chargés ✓")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/801 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

Invalid model-index. Not loading eval results into CardData.


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

[transformers] XLMRobertaForTokenClassification LOAD REPORT from: magistermilitum/roberta-multilingual-medieval-ner
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.weight | UNEXPECTED |  | 
roberta.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Invalid model-index. Not loading eval results into CardData.


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

[transformers] XLMRobertaForTokenClassification LOAD REPORT from: magistermilitum/roberta-multilingual-medieval-ner
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.weight | UNEXPECTED |  | 
roberta.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Invalid model-index. Not loading eval results into CardData.
Invalid model-index. Not loading eval results into CardData.


  Modèles chargés ✓


In [6]:
# ════════════════════════════════════════════════════════════════
# Cell 4 — Fonctions BIO (gold et prédictions)
# BIO 序列构建函数（gold 和预测）
# ════════════════════════════════════════════════════════════════

def build_char_bio(raw, spans, label_map):
    """Gold BIO au niveau caractère — auteur → PERS."""
    char_labels = ["O"] * len(raw)
    for span in spans:
        mapped = label_map.get(span["label"])
        if mapped is None:
            continue
        idx = raw.find(span["text"])
        if idx == -1:
            continue
        char_labels[idx] = f"B-{mapped}"
        for i in range(idx + 1, idx + len(span["text"])):
            char_labels[i] = f"I-{mapped}"
    return char_labels


def build_char_bio_ouvrage_pers(raw, spans):
    """
    Gold BIO pour les personnes dans les ouvrages.
    改动2 gold 侧：从 contains_persons 字段提取人名，在原文中定位标注为 PERS。
    """
    char_labels = ["O"] * len(raw)
    for span in spans:
        if span.get("label_detail") != "ouvrage_with_pers":
            continue
        ouvrage_idx = raw.find(span["text"])
        if ouvrage_idx == -1:
            continue
        ouvrage_sub = raw[ouvrage_idx: ouvrage_idx + len(span["text"])]
        for person in span.get("contains_persons", []):
            p_idx = ouvrage_sub.find(person)
            if p_idx == -1:
                continue
            abs_idx = ouvrage_idx + p_idx
            char_labels[abs_idx] = "B-PERS"
            for i in range(abs_idx + 1, abs_idx + len(person)):
                char_labels[i] = "I-PERS"
    return char_labels


def char_to_word_bio(raw, char_labels):
    """BIO caractère → BIO mot (premier caractère de chaque mot)."""
    bio, pos = [], 0
    for word in raw.split():
        idx = raw.find(word, pos)
        bio.append("O" if idx == -1 else char_labels[idx])
        pos = (idx + len(word)) if idx != -1 else (pos + len(word))
    return bio


def pred_to_char_bio(raw, predictions, keep_labels):
    """Prédictions modèle → BIO caractère (filtre par label)."""
    char_pred = ["O"] * len(raw)
    for pred in predictions:
        if pred["entity_group"] not in keep_labels:
            continue
        start = pred["start"]
        end   = min(pred["end"], len(raw))
        if start >= len(raw):
            continue
        char_pred[start] = f"B-{pred['entity_group']}"
        for i in range(start + 1, end):
            char_pred[i] = f"I-{pred['entity_group']}"
    return char_pred


In [7]:
# ════════════════════════════════════════════════════════════════
# Cell 5 — Fonctions overlap et reconstruction de texte
# 改动1 + 改动2：空格修复函数 + 两列互斥过滤函数
# ════════════════════════════════════════════════════════════════

# ── 改动1 — Reconstruction texte pour stratégie first ────────────────────────

def reconstruct_from_offsets(raw, pred):
    """
    改动1：first 策略 word 字段会拼接 subword 丢失空格。
    改为用 start/end 从原文切片，再 " ".join() 重建。
    """
    fragment = raw[pred.get("start", 0): min(pred.get("end", 0), len(raw))]
    return " ".join(fragment.split())


def get_pred_text(raw, pred, strategy):
    """改动1：only first 用 offset 重建，simple/max 直接用 word 字段。"""
    return reconstruct_from_offsets(raw, pred) if strategy == "first" else pred["word"]


# ── 改动2 — Overlap helpers (两列互斥) ───────────────────────────────────────

def spans_overlap(s1, e1, s2, e2):
    return s1 < e2 and s2 < e1


def get_auteur_intervals(raw, auteur_spans):
    """计算 gold auteur spans 在原文中的字符区间。"""
    intervals = []
    for span in auteur_spans:
        idx = raw.find(span["text"])
        if idx != -1:
            intervals.append((idx, idx + len(span["text"])))
    return intervals


def get_ouvrage_pers_intervals(raw, spans):
    """计算 gold ouvrage 里人名在原文中的字符区间。"""
    intervals = []
    for span in spans:
        if span.get("label_detail") != "ouvrage_with_pers":
            continue
        ouvrage_idx = raw.find(span["text"])
        if ouvrage_idx == -1:
            continue
        ouvrage_sub = raw[ouvrage_idx: ouvrage_idx + len(span["text"])]
        for person in span.get("contains_persons", []):
            p_idx = ouvrage_sub.find(person)
            if p_idx != -1:
                abs_start = ouvrage_idx + p_idx
                intervals.append((abs_start, abs_start + len(person)))
    return intervals


def split_preds_exclusive(raw, predictions, auteur_intervals, ouvrage_pers_intervals):
    """
    改动2 + Bug fix：将 PERS 预测严格分为两个互斥集合。
    - pred_auteur   : 与 gold auteur 位置重叠的预测
    - pred_ouvrage  : 与 gold ouvrage 人名位置重叠的预测（不在 pred_auteur 中）
    同一预测只属于一列，不会同时出现在两列。
    """
    pred_auteur, pred_ouvrage = [], []
    for pred in predictions:
        if pred["entity_group"] != "PERS":
            continue
        ps, pe = pred["start"], pred["end"]
        in_auteur = any(spans_overlap(ps, pe, a, b) for a, b in auteur_intervals)
        if in_auteur:
            pred_auteur.append(pred)
        else:
            in_ouvrage = any(spans_overlap(ps, pe, a, b) for a, b in ouvrage_pers_intervals)
            if in_ouvrage:
                pred_ouvrage.append(pred)
    return pred_auteur, pred_ouvrage


In [11]:
# ════════════════════════════════════════════════════════════════
# Cell 6 — Inférence sur les 3 stratégies
# 三种策略推理，构建 BIO 序列，收集详细结果
# ════════════════════════════════════════════════════════════════

LABEL_MAP = {"auteur": "PERS"}
strategy_results = {}

for strategy, ner_pipe in ner_pipes.items():
    print(f"\n  aggregation_strategy='{strategy}'...")

    all_gold_auteur          = []
    all_pred_pers_all        = []   # pred PERS all vs gold auteur
    all_pred_auteur_excl     = []   # pred auteur exclusif vs gold auteur
    all_gold_ouvrage_pers    = []
    all_pred_ouvrage_pers    = []
    details = []

    for entry in test_data:
        raw          = entry["raw"]
        spans        = entry.get("spans", [])
        auteur_spans = [s for s in spans if s["label"] == "auteur"]

        # Gold BIO
        gold_bio_a = char_to_word_bio(raw, build_char_bio(raw, spans, LABEL_MAP))
        gold_bio_o = char_to_word_bio(raw, build_char_bio_ouvrage_pers(raw, spans))

        # Inférence
        try:
            predictions = ner_pipe(raw)
        except Exception as e:
            _log(f"  [WARN] #{entry['entry_id']}: {e}")
            predictions = []

        # Calcul des intervalles gold
        auteur_ivs       = get_auteur_intervals(raw, auteur_spans)
        ouvrage_pers_ivs = get_ouvrage_pers_intervals(raw, spans)

        # Split exclusif
        pred_a_list, pred_o_list = split_preds_exclusive(
            raw, predictions, auteur_ivs, ouvrage_pers_ivs
        )

        # BIO 1 — pred PERS all (toutes les prédictions PERS)
        char_pred_all    = pred_to_char_bio(raw, predictions, {"PERS"})
        pred_bio_all     = char_to_word_bio(raw, char_pred_all)

        # BIO 2 — pred auteur exclusif
        char_pred_a_excl = pred_to_char_bio(raw, pred_a_list, {"PERS"})
        pred_bio_a_excl  = char_to_word_bio(raw, char_pred_a_excl)

        # BIO 3 — pred ouvrage_pers exclusif
        char_pred_o      = pred_to_char_bio(raw, pred_o_list, {"PERS"})
        pred_bio_o       = char_to_word_bio(raw, char_pred_o)

        all_gold_auteur.append(gold_bio_a)
        all_pred_pers_all.append(pred_bio_all)
        all_pred_auteur_excl.append(pred_bio_a_excl)
        all_gold_ouvrage_pers.append(gold_bio_o)
        all_pred_ouvrage_pers.append(pred_bio_o)

        # Textes lisibles
        gold_auteurs_txt   = [s["text"] for s in auteur_spans]
        gold_ouvrage_p_txt = [
            p for s in spans
            if s.get("label_detail") == "ouvrage_with_pers"
            for p in s.get("contains_persons", [])
        ]
        all_pers_txt   = [get_pred_text(raw, p, strategy)
                          for p in predictions if p["entity_group"] == "PERS"]
        pred_a_txt     = [get_pred_text(raw, p, strategy) for p in pred_a_list]
        pred_o_txt     = [get_pred_text(raw, p, strategy) for p in pred_o_list]

        details.append({
            "entry_id":             entry["entry_id"],
            "raw":                  raw,
            "gold_auteur":          gold_auteurs_txt,
            "gold_ouvrage_persons": gold_ouvrage_p_txt,
            "pred_pers_all":        all_pers_txt,
            "pred_pers_auteur":     pred_a_txt,
            "pred_pers_ouvrage":    pred_o_txt,
        })
        _log(
            f"  [{strategy}] #{entry['entry_id']:4d}"
            f" | pred_all={all_pers_txt}"
            f" | pred→auteur={pred_a_txt}"
            f" | pred→ouvr={pred_o_txt}"
            f" | gold_auteur={gold_auteurs_txt}"
            f" | gold_ouvr_pers={gold_ouvrage_p_txt}"
        )

    strategy_results[strategy] = {
        "all_gold_auteur":       all_gold_auteur,
        "all_pred_pers_all":     all_pred_pers_all,
        "all_pred_auteur_excl":  all_pred_auteur_excl,
        "all_gold_ouvrage_pers": all_gold_ouvrage_pers,
        "all_pred_ouvrage_pers": all_pred_ouvrage_pers,
        "details":               details,
    }

print("\n  Inférence terminée ✓")

[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset



  aggregation_strategy='simple'...
  [simple] #1298 | pred_all=[] | pred→auteur=[] | pred→ouvr=[] | gold_auteur=['da Cascia Fra Simone'] | gold_ouvr_pers=[]
  [simple] # 276 | pred_all=[] | pred→auteur=[] | pred→ouvr=[] | gold_auteur=['Hieronymi', 'Origenis'] | gold_ouvr_pers=[]
  [simple] # 297 | pred_all=[] | pred→auteur=[] | pred→ouvr=[] | gold_auteur=[] | gold_ouvr_pers=[]
  [simple] # 333 | pred_all=['Ioannis', 'Gu', 'al', 'ber', 'Bernard'] | pred→auteur=[] | pred→ouvr=['Ioannis', 'Gu', 'al', 'ber', 'Bernard'] | gold_auteur=[] | gold_ouvr_pers=['Ioannis Gualberti', 'Bernardi']
  [simple] # 338 | pred_all=['Leonis Urbe'] | pred→auteur=['Leonis Urbe'] | pred→ouvr=[] | gold_auteur=['Leonis Urbevetani'] | gold_ouvr_pers=[]
  [simple] # 351 | pred_all=['Basil', 'Mar', 'silii Fi'] | pred→auteur=['Basil', 'Mar', 'silii Fi'] | pred→ouvr=[] | gold_auteur=['Basilii', 'Marsilii Ficini'] | gold_ouvr_pers=[]
  [simple] #1512 | pred_all=[] | pred→auteur=[] | pred→ouvr=[] | gold_auteur=[] | gol

In [12]:
# ════════════════════════════════════════════════════════════════
# Cell 7 — Calcul des métriques (trois F1)
# 三套 F1：F1_pers_all / F1_auteur_excl / F1_ouvrage_pers
# ════════════════════════════════════════════════════════════════

_log("=" * 80)
_log(f"{'Stratégie':<10} {'F1_pers_all':>12} {'F1_auteur_excl':>16} {'F1_ouvrage_pers':>17}")
_log(f"{'':10} {'P':>6}{'R':>6}{'F1':>6}  {'P':>6}{'R':>6}{'F1':>6}  {'P':>6}{'R':>6}{'F1':>6}")
_log("-" * 80)

metrics_by_strategy = {}

for strategy, res in strategy_results.items():

    # F1_pers_all — pred PERS (all) vs gold auteur
    f1_pa  = f1_score(res["all_gold_auteur"], res["all_pred_pers_all"], zero_division=0)
    pre_pa = precision_score(res["all_gold_auteur"], res["all_pred_pers_all"], zero_division=0)
    rec_pa = recall_score(res["all_gold_auteur"], res["all_pred_pers_all"], zero_division=0)

    # F1_auteur_excl — pred auteur (exclusif) vs gold auteur
    f1_a  = f1_score(res["all_gold_auteur"], res["all_pred_auteur_excl"], zero_division=0)
    pre_a = precision_score(res["all_gold_auteur"], res["all_pred_auteur_excl"], zero_division=0)
    rec_a = recall_score(res["all_gold_auteur"], res["all_pred_auteur_excl"], zero_division=0)

    # F1_ouvrage_pers — pred ouvrage (exclusif) vs gold ouvrage persons
    f1_o  = f1_score(res["all_gold_ouvrage_pers"], res["all_pred_ouvrage_pers"], zero_division=0)
    pre_o = precision_score(res["all_gold_ouvrage_pers"], res["all_pred_ouvrage_pers"], zero_division=0)
    rec_o = recall_score(res["all_gold_ouvrage_pers"], res["all_pred_ouvrage_pers"], zero_division=0)

    metrics_by_strategy[strategy] = {
        "pers_all":      {"f1": round(f1_pa,4), "precision": round(pre_pa,4), "recall": round(rec_pa,4)},
        "auteur_excl":   {"f1": round(f1_a,4),  "precision": round(pre_a,4),  "recall": round(rec_a,4)},
        "ouvrage_pers":  {"f1": round(f1_o,4),  "precision": round(pre_o,4),  "recall": round(rec_o,4)},
    }

    _log(
        f"{strategy:<10}"
        f" {pre_pa:>6.3f}{rec_pa:>6.3f}{f1_pa:>6.3f}"
        f"  {pre_a:>6.3f}{rec_a:>6.3f}{f1_a:>6.3f}"
        f"  {pre_o:>6.3f}{rec_o:>6.3f}{f1_o:>6.3f}"
    )

    _log(f"\n  [{strategy}] rapport pers_all (pred PERS all vs gold auteur) :")
    _log(classification_report(res["all_gold_auteur"], res["all_pred_pers_all"], zero_division=0))
    _log(f"  [{strategy}] rapport auteur_excl (pred auteur exclusif vs gold auteur) :")
    _log(classification_report(res["all_gold_auteur"], res["all_pred_auteur_excl"], zero_division=0))
    _log(f"  [{strategy}] rapport ouvrage_pers :")
    _log(classification_report(res["all_gold_ouvrage_pers"], res["all_pred_ouvrage_pers"], zero_division=0))

    # Détail par entrée
    _log(f"\n  {'─'*60}")
    _log(f"  [{strategy}] Détail par entrée")
    _log(f"  {'─'*60}")
    for d in res["details"]:
        _log(f"\n  #{d['entry_id']}  {d['raw'][:70]}...")
        _log(f"    pred PERS (all)       : {d['pred_pers_all']}")
        _log(f"    → pred auteur (excl)  : {d['pred_pers_auteur']}")
        _log(f"    → pred ouvr. pers     : {d['pred_pers_ouvrage']}")
        _log(f"    gold auteur           : {d['gold_auteur']}")
        _log(f"    gold ouvr. pers       : {d['gold_ouvrage_persons']}")
    _log(f"\n  {'─'*60}\n")

Stratégie   F1_pers_all   F1_auteur_excl   F1_ouvrage_pers
                P     R    F1       P     R    F1       P     R    F1
--------------------------------------------------------------------------------
simple      0.455 0.500 0.476   0.833 0.500 0.625   0.333 0.333 0.333

  [simple] rapport pers_all (pred PERS all vs gold auteur) :
              precision    recall  f1-score   support

        PERS       0.45      0.50      0.48        10

   micro avg       0.45      0.50      0.48        10
   macro avg       0.45      0.50      0.48        10
weighted avg       0.45      0.50      0.48        10

  [simple] rapport auteur_excl (pred auteur exclusif vs gold auteur) :
              precision    recall  f1-score   support

        PERS       0.83      0.50      0.62        10

   micro avg       0.83      0.50      0.62        10
   macro avg       0.83      0.50      0.62        10
weighted avg       0.83      0.50      0.62        10

  [simple] rapport ouvrage_pers :
       

In [14]:
# ════════════════════════════════════════════════════════════════
# Cell 8 — Sauvegarde BIO + JSON + Markdown
# 保存 BIO 序列、JSON 结果、Markdown 报告
# ════════════════════════════════════════════════════════════════

# ── BIO sequences ──
for strategy, res in strategy_results.items():
    bio_path = os.path.join(LOG_DIR, f"bio_sequences_v3_{strategy}.txt")
    with open(bio_path, "w", encoding="utf-8") as f:
        for i, entry in enumerate(test_data):
            f.write(f"\n=== #{entry['entry_id']} ===\n")
            f.write(f"raw              : {entry['raw']}\n")
            f.write(f"gold_auteur      : {res['all_gold_auteur'][i]}\n")
            f.write(f"pred_pers_all    : {res['all_pred_pers_all'][i]}\n")
            f.write(f"pred_auteur_excl : {res['all_pred_auteur_excl'][i]}\n")
            f.write(f"gold_ouvr_pers   : {res['all_gold_ouvrage_pers'][i]}\n")
            f.write(f"pred_ouvr_pers   : {res['all_pred_ouvrage_pers'][i]}\n")
    _log(f"[BIO] [{strategy}] → {bio_path}")

# ── JSON ──
class NumpyEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.integer): return int(obj)
        if isinstance(obj, np.floating): return float(obj)
        if isinstance(obj, np.ndarray):  return obj.tolist()
        return super().default(obj)

results = {
    "model":     MODEL_NAME,
    "run":       "Run 1 v3 — trois F1 : pers_all / auteur_excl / ouvrage_pers",
    "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "test_entries": len(test_data),
    "ameliorations": {
        "1": "first : reconstruction texte depuis offsets avec ' '.join()",
        "2": "pred_auteur / pred_ouvrage exclusifs via split_preds_exclusive()",
        "3": "trois F1 : F1_pers_all / F1_auteur_excl / F1_ouvrage_pers",
        "bugfix": "v2 → v3 : pred_auteur strictement exclusif",
    },
    "metrics_by_strategy": metrics_by_strategy,
    "details_par_strategie": {s: res["details"] for s, res in strategy_results.items()},
}

out_path = os.path.join(LOG_DIR, "test_results_run1_v3.json")
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2, cls=NumpyEncoder)
_log(f"[JSON] → {out_path}")

# ── Markdown ──
def classify_error(gold_auteurs, pred_all):
    if not gold_auteurs and not pred_all: return ["correct_vide"]
    errors = []
    if gold_auteurs and not pred_all: errors.append("faux_negatif")
    if pred_all and not gold_auteurs:  errors.append("faux_positif")
    if gold_auteurs and pred_all:
        matched = any(g in p or p in g or any(t in p for t in g.split())
                      for g in gold_auteurs for p in pred_all)
        if matched:
            exact = any(g.strip() == p.strip() for g in gold_auteurs for p in pred_all)
            errors.append("correct_exact" if exact else "erreur_frontiere")
        else:
            errors.append("faux_negatif")
            if pred_all: errors.append("faux_positif")
    return errors or ["correct_exact"]

details_s = strategy_results["simple"]["details"]
type_counts = {"correct_exact":[], "correct_vide":[], "faux_negatif":[],
               "faux_positif":[], "erreur_frontiere":[]}
for d in details_s:
    for e in classify_error(d["gold_auteur"], d["pred_pers_all"]):
        if e in type_counts: type_counts[e].append(d["entry_id"])

md = []
md.append("# Rapport Run 1 v3\n")
md.append(f"**Modèle** : `{MODEL_NAME}`  ")
md.append(f"**Date** : {datetime.now().strftime('%Y-%m-%d %H:%M')}  ")
md.append(f"**Données** : `test_run1.json` ({len(test_data)} entrées)\n")
md.append("## Métriques\n")
md.append("| Stratégie | F1_pers_all | P | R | F1_auteur_excl | P | R | F1_ouvrage_pers | P | R |")
md.append("|---|---|---|---|---|---|---|---|---|---|")
for s, m in metrics_by_strategy.items():
    pa = m["pers_all"]
    a  = m["auteur_excl"]
    o  = m["ouvrage_pers"]
    md.append(
        f"| {s} | {pa['f1']:.4f} | {pa['precision']:.4f} | {pa['recall']:.4f}"
        f" | {a['f1']:.4f} | {a['precision']:.4f} | {a['recall']:.4f}"
        f" | {o['f1']:.4f} | {o['precision']:.4f} | {o['recall']:.4f} |"
    )
md.append("\n## Analyse des erreurs (stratégie simple)\n")
labels_fr = {
    "correct_exact":    "Correct (exact)",
    "correct_vide":     "Correct (rien attendu)",
    "faux_negatif":     "Faux négatif",
    "faux_positif":     "Faux positif",
    "erreur_frontiere": "Erreur de frontière",
}
md.append("| Type | N | entry_ids |\n|---|---|---|")
for t, ids in type_counts.items():
    if ids:
        md.append(f"| {labels_fr[t]} | {len(ids)} | {', '.join(f'#{i}' for i in ids)} |")
md.append("\n## Détails par entrée (stratégie simple)\n")
for d in details_s:
    md.append(f"**#{d['entry_id']}** `{d['raw'][:80]}...`")
    md.append(f"- pred PERS (all)      : {d['pred_pers_all']}")
    md.append(f"- pred auteur (excl)   : {d['pred_pers_auteur']}")
    md.append(f"- pred ouvr. pers      : {d['pred_pers_ouvrage']}")
    md.append(f"- gold auteur          : {d['gold_auteur']}")
    md.append(f"- gold ouvr. pers      : {d['gold_ouvrage_persons']}\n")

md_path = os.path.join(LOG_DIR, "error_analysis_run1_v3.md")
with open(md_path, "w", encoding="utf-8") as f:
    f.write("\n".join(md) + "\n")
_log(f"[Markdown] → {md_path}")

_save_log()
print("\n✓ Tout sauvegardé dans", LOG_DIR)

[BIO] [simple] → /content/run1_v3/bio_sequences_v3_simple.txt
[BIO] [first] → /content/run1_v3/bio_sequences_v3_first.txt
[BIO] [max] → /content/run1_v3/bio_sequences_v3_max.txt
[JSON] → /content/run1_v3/test_results_run1_v3.json
[Markdown] → /content/run1_v3/error_analysis_run1_v3.md

✓ Tout sauvegardé dans /content/run1_v3
